# 00 · Explore the seven datasets and make network/availability figures

**Start here.** For current HPC demonstration leave `config/active_profile.json` at `profile: hpc`. For outside users, copy `config/user_paths.example.json` to `config/user_paths.json`, edit the seven paths (and optional external radar/WVD sources), then set `profile: user` **once** in `config/active_profile.json`. Nothing else needs to be edited.

Outputs: network map, frequency/length distribution, five-sensor native availability and monthly CSV/PNG. The next notebook checks that this stage completed. This is a read-only workflow; figures are reconstructions, not automatically identical in style to manuscript originals.

**About this notebook.** Needs: all seven NetCDF files, configured once in `config/user_paths.json`. Parameters: `RUN_NATIVE_AVAILABILITY`. Makes: network map, CML characteristics, monthly and native availability tables and figures. Cost: minutes. Fidelity: availability uses the unchanged original script and reproduces the paper numbers exactly; the network and characteristics plots are reviewer figures. Details: `docs/NOTEBOOK_GUIDE.md`.

In [4]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


Profile: hpc Data config: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/config/hpc_source.example.json Output: /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow


In [5]:
from essd_cz.catalog import inspect_all
from essd_cz.notebook_figures import plot_network, plot_characteristics, monthly_from_native_csv
from essd_cz.workflows import availability
print('Seven configured inputs:')
for name,p in project.paths.items(): print(name,':',p)
print('External inputs (optional until later stages):')
for name,p in project.extras.items(): print(name,':',p)

Seven configured inputs:
raw : /home/jovyan/ESSD_publication_dataset/CZ_CML_RAW_1MIN_20200101_20211231.nc
qc : /home/jovyan/ESSD_publication_dataset/CZ_CML_PREPROCESSED_1MIN_20200101_20211231.nc
rain : /home/jovyan/ESSD_publication_dataset/cmlonly_graf_itu2005_pastorek/CZ_CML_RAINRATE_GRAF_ITU2005_PASTOREK_5MIN_20200101_20211231.nc
gauge : /home/jovyan/ESSD_publication_dataset/CZ_RAIN_GAUGE_10MIN_20200101_20211231.nc
radar : /home/jovyan/ESSD_publication_dataset/CZ_RADAR_CMLPATH_5MIN_20200101_20211231.nc
temperature : /home/jovyan/opendata.chmi.cz/processed/CHMI_10MIN_TEMPERATURE_2021.nc
humidity : /home/jovyan/opendata.chmi.cz/processed/CHMI_10MIN_HUMIDITY_2021.nc
External inputs (optional until later stages):
native_radklim_root : /home/jovyan/dwd_radolan/2021/suplement
historical_event_cache : /home/jovyan/ESSD_publication_dataset/cmlonly_graf_nearby_comparison/apr_oct_2021/article_style/multiple_events_radar/selected_instances_cache.csv
wvd_v6_nc : /home/jovyan/ESSD_publication_dat

In [6]:
FIGURE_DPI = 200
RUN_NATIVE_AVAILABILITY = True  # reads April–October only; on HPC took about 90 s
EXPLORATION = project.stage_dir('00_exploration')
network,station_counts = plot_network(project.paths, EXPLORATION/'network', dpi=FIGURE_DPI)
characteristics,cml_counts = plot_characteristics(project.paths, EXPLORATION/'cml', dpi=FIGURE_DPI)
print('Network:',station_counts,'Frequency and length:',cml_counts)
if RUN_NATIVE_AVAILABILITY:
    availability(project.paths, EXPLORATION/'native_availability', dpi=FIGURE_DPI)
    monthly_png,monthly_csv,monthly = monthly_from_native_csv(EXPLORATION/'native_availability',EXPLORATION/'monthly_availability',dpi=FIGURE_DPI)
    print(monthly)
project.register('exploration_complete',characteristics,'reviewer network/frequency/path-length plots and optional availability')

Network: {'cml_paths': 2219, 'gauge': 424, 'temperature': 251, 'humidity': 247} Frequency and length: {'directional_pairs': 4438, 'eband_pairs': 1804}
ORIGINAL COMMAND: /home/jovyan/.venvs/pr13_xarray_2026_04/bin/python /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/original/code/figures/availability_native_timestep_percentage.py --cml-file /home/jovyan/ESSD_publication_dataset/CZ_CML_PREPROCESSED_1MIN_20200101_20211231.nc --radar-file /home/jovyan/ESSD_publication_dataset/CZ_RADAR_CMLPATH_5MIN_20200101_20211231.nc --gauge-file /home/jovyan/ESSD_publication_dataset/CZ_RAIN_GAUGE_10MIN_20200101_20211231.nc --temperature-file /home/jovyan/opendata.chmi.cz/processed/CHMI_10MIN_TEMPERATURE_2021.nc --humidity-file /home/jovyan/opendata.chmi.cz/processed/CHMI_10MIN_HUMIDITY_2021.nc --out-dir /home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/00_exploration/native_availability --dpi 200
NATIVE-TIMESTEP SENSOR AVA

PosixPath('/home/jovyan/ESSD_publication_dataset/ESSD_CZ_2021_CONNECTED_WALKTHROUGH_V2/results/connected_workflow/00_exploration/cml/reviewer_f03_cml_characteristics.png')